[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/Quantlets/Ch_10/TSA_ch10_markov_gdp/TSA_ch10_markov_gdp.ipynb)

# TSA_ch10_markov_gdp

Markov switching (Hamilton 1989) for GDP growth: the MS-AR(4) on 1951-1984 with two local maxima of the likelihood; a switching mean on US real GDP growth 1947-2019 against the NBER dates, extended to 2026; filtered against smoothed probabilities in 2008; specifications that find other regimes (variance, 2020); Romanian GDP growth regimes with switching mean and variance.

*Time Series Analysis (TSA), Chapter 10: State space models, Kalman filter and Markov switching. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/TSA/Chapter_10'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# The arch package (GARCH estimation) is not preinstalled in Google Colab: pip install arch
import importlib.util, subprocess, sys
if importlib.util.find_spec('arch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'arch'])

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import tsa_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the TSA repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import statsmodels.api as sm

SEED = 2026

GDP_RO = ('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')

HP_LAMBDA = 1600

HAM_H, HAM_P = 8, 4

DIFFUSE = 10000000.0

NILE_GAPS = [(1891, 1910), (1931, 1950)]

HAMILTON_SAMPLE = ('1951-04-01', '1984-10-01')

MS_SAMPLE = ('1947-04-01', '2019-10-01')

COINCIDENT = ['INDPRO', 'PAYEMS', 'W875RX1', 'CMRMTSPL']

COINC_NAMES = {'INDPRO': 'industrial production', 'PAYEMS': 'payroll employment', 'W875RX1': 'real income less transfers', 'CMRMTSPL': 'real manufacturing and trade sales'}

TVP_PAIR = ('bet', 'stoxx50')

TVP_START = '2005-01-01'

ROLL = 52

SEARCH = 20

SHADE = st.Amber

def nile():
    """Annual flow of the Nile at Aswan, 1871-1970, 10^8 cubic metres (statsmodels data set)."""
    s = load_statsmodels('nile')
    s.index = s.index.year
    return s.astype(float)


def ro_gdp():
    """Log Romanian real GDP x 100 (quarterly, seasonally and calendar adjusted, Eurostat namq_10_gdp)."""
    x = read_eurostat(*GDP_RO)
    return (100 * np.log(x)).rename('ro_gdp')


def us_gdp_growth():
    """US real GDP growth, quarter on quarter, 100 ln(GDP_t / GDP_{t-1}), in % (FRED GDPC1)."""
    g = read_fred('GDPC1').dropna()
    return (100 * np.log(g).diff()).dropna().rename('us_gdp')


def nber(freq='QS'):
    """NBER recession indicator (FRED USREC): 1 if any month of the period is in a recession."""
    r = read_fred('USREC').dropna()
    return r.resample(freq).max() if freq != 'MS' else r


def weekly_returns(name, start='2000-01-01'):
    """Weekly log returns in % (Friday closes, or the last trading day of the week)."""
    p = load_close(name, start)
    w = p.resample('W-FRI').last().dropna()
    return (100 * np.log(w).diff()).dropna().rename(name)


def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def shade(ax, rec, label='NBER recessions'):
    """Shade the periods with rec == 1 (a 0/1 series with a DatetimeIndex)."""
    rec = rec.fillna(0).astype(int)
    on, first = None, True
    idx = rec.index
    step = idx[1] - idx[0] if len(idx) > 1 else pd.Timedelta(days=90)
    for t, v in rec.items():
        if v and on is None:
            on = t
        if not v and on is not None:
            ax.axvspan(on, t, color=SHADE, alpha=0.28, lw=0, label=label if first else '_')
            on, first = None, False
    if on is not None:
        ax.axvspan(on, idx[-1] + step, color=SHADE, alpha=0.28, lw=0, label=label if first else '_')


def kalman_filter(y, Z, T, H, Q, a1, P1, d=0):
    """Kalman filter for  y_t = Z_t a_t + eps_t, eps_t ~ N(0, H);  a_{t+1} = T a_t + eta_t, eta_t ~ N(0, Q).
    Z: (m,) or (n, m) for a time-varying Z_t; missing y_t (NaN) are skipped in the update step.
    Returns predicted a_t|t-1, P_t|t-1, filtered a_t|t, P_t|t, innovations v_t, their variances F_t, the gains
    K_t = P_t|t-1 Z_t' / F_t and the log-likelihood of the prediction-error decomposition (first d observations
    excluded: the diffuse part)."""
    y = np.asarray(y, float)
    n, m = len(y), len(np.atleast_1d(a1))
    Z = np.asarray(Z, float)
    Zt = np.tile(Z, (n, 1)) if Z.ndim == 1 else Z
    T, Q = np.atleast_2d(T).astype(float), np.atleast_2d(Q).astype(float)
    a, P = np.atleast_1d(a1).astype(float).copy(), np.atleast_2d(P1).astype(float).copy()
    ap, Pp, af, Pf = np.zeros((n, m)), np.zeros((n, m, m)), np.zeros((n, m)), np.zeros((n, m, m))
    v, F, K = np.full(n, np.nan), np.full(n, np.nan), np.zeros((n, m))
    ll, nobs = 0.0, 0
    for t in range(n):
        ap[t], Pp[t] = a, P
        z = Zt[t]
        if np.isnan(y[t]):
            af[t], Pf[t] = a, P                                  # no observation: no update
        else:
            v[t] = y[t] - z @ a
            F[t] = z @ P @ z + H
            K[t] = P @ z / F[t]
            af[t] = a + K[t] * v[t]
            Pf[t] = P - np.outer(K[t], z @ P)
            if t >= d:
                ll += -0.5 * (np.log(2 * np.pi) + np.log(F[t]) + v[t] ** 2 / F[t])
                nobs += 1
        a = T @ af[t]                                            # prediction step
        P = T @ Pf[t] @ T.T + Q
    return {'a_pred': ap, 'P_pred': Pp, 'a_filt': af, 'P_filt': Pf, 'v': v, 'F': F, 'K': K, 'loglik': ll,
            'nobs': nobs, 'a_next': a, 'P_next': P, 'Z': Zt, 'T': T, 'H': H, 'Q': Q}


def rts_smoother(kf):
    """Rauch-Tung-Striebel (1965) smoother: a_t|n = a_t|t + J_t (a_t+1|n - a_t+1|t), J_t = P_t|t T' P_t+1|t^-1,
    P_t|n = P_t|t + J_t (P_t+1|n - P_t+1|t) J_t'."""
    af, Pf, ap, Pp, T = kf['a_filt'], kf['P_filt'], kf['a_pred'], kf['P_pred'], kf['T']
    n = len(af)
    a_s, P_s = af.copy(), Pf.copy()
    for t in range(n - 2, -1, -1):
        J = Pf[t] @ T.T @ np.linalg.inv(Pp[t + 1])
        a_s[t] = af[t] + J @ (a_s[t + 1] - ap[t + 1])
        P_s[t] = Pf[t] + J @ (P_s[t + 1] - Pp[t + 1]) @ J.T
    return a_s, P_s


def disturbance_smoother(kf):
    """Smoothed disturbances (Durbin and Koopman 2012, Sec. 2.5 and 4.5) for a univariate observation: backward
    recursions r_t-1 = Z' v_t / F_t + L_t' r_t, N_t-1 = Z' Z / F_t + L_t' N_t L_t with L_t = T - T K_t Z_t.
    Returns the standardised observation residuals u_t / sqrt(D_t) and state residuals r_t / sqrt(diag N_t)
    (the auxiliary residuals used to detect outliers and level breaks)."""
    Zt, T, v, F, K = kf['Z'], kf['T'], kf['v'], kf['F'], kf['K']
    n, m = Zt.shape
    r, N = np.zeros(m), np.zeros((m, m))
    obs_std, state_std = np.full(n, np.nan), np.full((n, m), np.nan)
    for t in range(n - 1, -1, -1):
        with np.errstate(invalid='ignore', divide='ignore'):
            state_std[t] = r / np.sqrt(np.diag(N))               # r_t and N_t refer to eta_t (between t and t+1)
        z = Zt[t]
        if np.isnan(v[t]):
            r, N = T.T @ r, T.T @ N @ T
            continue
        Kt = T @ K[t]                                            # gain of a_t+1|t
        L = T - np.outer(Kt, z)
        u = v[t] / F[t] - Kt @ r
        D = 1 / F[t] + Kt @ N @ Kt
        obs_std[t] = u / np.sqrt(D)
        r = z * v[t] / F[t] + L.T @ r
        N = np.outer(z, z) / F[t] + L.T @ N @ L
    return obs_std, state_std


def local_level_filter(y, s2_eps, s2_eta, a1=0.0, P1=DIFFUSE):
    """Local level (random walk plus noise): y_t = mu_t + eps_t, mu_t+1 = mu_t + eta_t."""
    return kalman_filter(y, [1.0], [[1.0]], s2_eps, [[s2_eta]], [a1], [[P1]], d=1)


def local_level_ml(y):
    """Maximum likelihood of (sigma2_eps, sigma2_eta) by the prediction-error decomposition (diffuse start)."""
    y = np.asarray(y, float)
    v0 = np.nanvar(np.diff(y[~np.isnan(y)]))
    f = lambda th: -local_level_filter(y, np.exp(th[0]), np.exp(th[1]))['loglik']
    best = min((optimize.minimize(f, x0, method='Nelder-Mead', options={'xatol': 1e-8, 'fatol': 1e-8, 'maxiter': 4000})
                for x0 in ([np.log(v0 / 2), np.log(v0 / 20)], [np.log(v0 / 20), np.log(v0 / 2)])), key=lambda o: o.fun)
    s2e, s2n = np.exp(best.x)
    kf = local_level_filter(y, s2e, s2n)
    n_par, n_obs = 2, kf['nobs']
    return {'s2_eps': float(s2e), 's2_eta': float(s2n), 'q': float(s2n / s2e), 'loglik': float(kf['loglik']),
            'aic': float(-2 * kf['loglik'] + 2 * n_par), 'bic': float(-2 * kf['loglik'] + np.log(n_obs) * n_par), 'kf': kf}


def steady_state(q):
    """Steady state of the local level filter with signal-to-noise ratio q = sigma2_eta / sigma2_eps:
    predicted variance P = sigma2_eps (q + sqrt(q^2 + 4q)) / 2 and gain K = P / (P + sigma2_eps)."""
    p = (q + np.sqrt(q ** 2 + 4 * q)) / 2
    return {'P': float(p), 'K': float(p / (p + 1)), 'Pf': float(p / (p + 1))}


def q_from_alpha(alpha):
    """Signal-to-noise ratio of the local level model whose steady-state gain is the SES weight alpha:
    q = alpha^2 / (1 - alpha) (Muth 1960)."""
    return alpha ** 2 / (1 - alpha)


def profile_loglik(y, q):
    """Log-likelihood with sigma2_eps concentrated out, for a given q (filter with sigma2_eps = 1, sigma2_eta = q)."""
    kf = local_level_filter(y, 1.0, q)
    ok = ~np.isnan(kf['v'])
    ok[0] = False
    v, F = kf['v'][ok], kf['F'][ok]
    n = ok.sum()
    s2 = np.mean(v ** 2 / F)
    return float(-0.5 * n * (np.log(2 * np.pi) + 1 + np.log(s2)) - 0.5 * np.log(F).sum()), float(s2)


def ses_path(y, alpha, l0):
    """Simple exponential smoothing (Chapter 0): l_t = alpha y_t + (1 - alpha) l_t-1; the forecast of y_t+1 is l_t."""
    l, out = l0, []
    for x in y:
        out.append(l)
        l = alpha * x + (1 - alpha) * l
    return np.array(out)


def worked_example(y=(12.0, 11.0, 14.0), a1=10.0, P1=4.0, s2_eps=4.0, s2_eta=1.0):
    """The three-period local level example of the slides, step by step."""
    rows, a, P = [], a1, P1
    for t, yt in enumerate(y, 1):
        F = P + s2_eps
        K = P / F
        v = yt - a
        af, Pf = a + K * v, P * (1 - K)
        rows.append({'t': t, 'y': yt, 'a': a, 'P': P, 'F': F, 'K': K, 'v': v, 'af': af, 'Pf': Pf})
        a, P = af, Pf + s2_eta
    ss = steady_state(s2_eta / s2_eps)
    return {'rows': rows, 'a_next': a, 'P_next': P, 'ss_P': ss['P'] * s2_eps, 'ss_K': ss['K'], 'q': s2_eta / s2_eps}


def ms_fit(model, seed=SEED, reps=SEARCH):
    np.random.seed(seed)
    return model.fit(search_reps=reps, disp=False)


def regime_order(r, key='const'):
    """Index of the regimes sorted by a parameter (e.g. the mean: recession first)."""
    names = r.model.param_names
    vals = [r.params[names.index(f'{key}[{k}]')] for k in range(r.model.k_regimes)]
    return list(np.argsort(vals))


def ms_summary(r, idx, low):
    """Parameters, transition probabilities, expected durations and ergodic probabilities of a two-regime fit,
    with regime `low` reported as regime 1 (recession / turbulent)."""
    P = r.regime_transition[:, :, 0]                      # P[i, j] = Pr(S_t = i | S_t-1 = j)
    high = 1 - low
    p11, p22 = float(P[low, low]), float(P[high, high])
    erg = (1 - p22) / (2 - p11 - p22)
    names = r.model.param_names
    out = {'p11': p11, 'p22': p22, 'dur1': 1 / (1 - p11), 'dur2': 1 / (1 - p22), 'ergodic1': float(erg),
           'loglik': float(r.llf), 'aic': float(r.aic), 'bic': float(r.bic), 'n': int(len(idx))}
    for nm, v in zip(names, r.params):
        if '[' in nm and not nm.startswith('p['):
            base, k = nm.split('[')
            k = int(k[:-1])
            out[f'{base}_{1 if k == low else 2}'] = float(v)
        elif not nm.startswith('p['):
            out[nm.replace('.', '_')] = float(v)
    return out


def concordance(prob, rec):
    """Share of periods where 'probability > 0.5' agrees with the NBER indicator; and the hits in NBER recessions."""
    prob, rec = pd.Series(prob), pd.Series(rec).reindex(prob.index).fillna(0)
    cls = (prob > 0.5).astype(int)
    return {'concord': float((cls == rec).mean()), 'hit': float(cls[rec == 1].mean()), 'false': float(cls[rec == 0].mean())}


def fig_ms_us(save_it=True):
    """Hamilton (1989): two-regime Markov-switching mean for US real GDP growth. The original specification
    (MS-AR(4), sample 1951Q2-1984Q4) and a switching mean without AR terms on 1947Q2-2019Q4; smoothed recession
    probabilities against the NBER dates, extended to 2026 with the parameters of 1947-2019."""
    g = us_gdp_growth()
    rec = nber('QS')
    x0 = g.loc[HAMILTON_SAMPLE[0]:HAMILTON_SAMPLE[1]]
    m0 = sm.tsa.MarkovAutoregression(x0.values, k_regimes=2, order=4, switching_ar=False)
    # two local maxima of the likelihood, reached from two starting points:
    # (a) a recession regime of several quarters (Hamilton 1989); (b) a regime of single sharp quarters
    r0 = m0.fit(start_params=[0.75, 0.10, -0.2, 1.3, 0.7, 0.0, 0.0, 0.0, 0.0], disp=False)
    r0b = m0.fit(start_params=[0.15, 0.05, -1.1, 1.0, 0.8, 0.3, 0.15, -0.15, -0.15], disp=False)
    low0, low0b = regime_order(r0)[0], regime_order(r0b)[0]
    x = g.loc[MS_SAMPLE[0]:MS_SAMPLE[1]]
    r = ms_fit(sm.tsa.MarkovRegression(x.values, k_regimes=2))
    low = regime_order(r)[0]
    xa = g.loc[MS_SAMPLE[0]:]
    ma = sm.tsa.MarkovRegression(xa.values, k_regimes=2)
    ra = ma.smooth(r.params)
    sp = pd.Series(ra.smoothed_marginal_probabilities[:, low], index=xa.index)
    fp = pd.Series(ra.filtered_marginal_probabilities[:, low], index=xa.index)
    fig, axes = plt.subplots(2, 1, figsize=(10, 5.4), sharex=True, gridspec_kw={'height_ratios': [1, 1]})
    shade(axes[0], rec.reindex(xa.index))
    axes[0].plot(xa.index, xa.values, color=st.MainBlue, lw=0.9, label='real GDP growth, % q/q')
    axes[0].axhline(0, color=st.DarkText, lw=0.6)
    axes[0].set_ylim(-4, 4)
    shade(axes[1], rec.reindex(xa.index), label='_')
    axes[1].plot(sp.index, sp.values, color=st.IDAred, lw=1.6, label='smoothed Pr(recession regime)')
    axes[1].axvline(pd.Timestamp('2020-01-01'), color=st.Purple, ls='--', lw=1.1, label='end of the estimation sample')
    axes[1].set_ylim(-0.02, 1.02)
    st.fig_legend_bottom(fig, ncol=4, y=0.02)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    save('tsa_ch10_ms_us', save_it)
    est = sp.loc[:MS_SAMPLE[1]]
    sp0 = pd.Series(r0.smoothed_marginal_probabilities[:, low0], index=x0.index[4:])
    s0, s = ms_summary(r0, x0.index[4:], low0), ms_summary(r, x.index, low)
    post = sp.loc['2020':]
    s0b = ms_summary(r0b, x0.index[4:], low0b)
    sp0b = pd.Series(r0b.smoothed_marginal_probabilities[:, low0b], index=x0.index[4:])
    return {'ham': s0, 'ham_conc': concordance(sp0, rec), 'ham_b': s0b, 'ham_b_conc': concordance(sp0b, rec), 'ext': s, 'ext_conc': concordance(est, rec),
            'ext_conc_f': concordance(fp.loc[:MS_SAMPLE[1]], rec), 'p2020q2': float(sp.loc['2020-04-01']),
            'p2020q3': float(sp.loc['2020-07-01']), 'post_max_after2021': float(post.loc['2021':].max()),
            'n_rec_q': int((rec.reindex(x.index) == 1).sum()), 'first': str(x.index[0].date()), 'last': str(x.index[-1].date()),
            'g2020q2': float(g.loc['2020-04-01']), 'sp_last': float(sp.iloc[-1]), 'last_d': str(sp.index[-1].date())}


def fig_ms_filtered(save_it=True):
    """Filtered (real-time) against smoothed recession probabilities around the 2008-2009 recession."""
    g = us_gdp_growth()
    rec = nber('QS')
    x = g.loc[MS_SAMPLE[0]:MS_SAMPLE[1]]
    r = ms_fit(sm.tsa.MarkovRegression(x.values, k_regimes=2))
    low = regime_order(r)[0]
    sp = pd.Series(r.smoothed_marginal_probabilities[:, low], index=x.index)
    fp = pd.Series(r.filtered_marginal_probabilities[:, low], index=x.index)
    pp = pd.Series(r.predicted_marginal_probabilities[:, low], index=x.index)
    w = slice('2005-01-01', '2012-10-01')
    fig, ax = plt.subplots(figsize=(10, 3.9))
    shade(ax, rec.reindex(x.index).loc[w])
    ax.plot(pp.loc[w].index, pp.loc[w].values, color=st.Teal, lw=1.2, ls=':', marker='.', label='predicted Pr(S_t = 1 | y_1..y_t-1)')
    ax.plot(fp.loc[w].index, fp.loc[w].values, color=st.MainBlue, lw=1.5, marker='o', ms=3.5, label='filtered Pr(S_t = 1 | y_1..y_t)')
    ax.plot(sp.loc[w].index, sp.loc[w].values, color=st.IDAred, lw=2, label='smoothed Pr(S_t = 1 | y_1..y_n)')
    ax.set_ylim(-0.02, 1.02)
    st.legend_outside_bottom(ax, ncol=2, y=-0.12)
    save('tsa_ch10_ms_filtered', save_it)
    q = lambda s, d: float(s.loc[d])
    return {'f2008q1': q(fp, '2008-01-01'), 's2008q1': q(sp, '2008-01-01'), 'f2008q3': q(fp, '2008-07-01'),
            's2008q3': q(sp, '2008-07-01'), 'f2008q4': q(fp, '2008-10-01'), 's2008q4': q(sp, '2008-10-01'),
            'f2009q2': q(fp, '2009-04-01'), 's2009q2': q(sp, '2009-04-01'), 'g2008q3': q(g, '2008-07-01'),
            'g2008q4': q(g, '2008-10-01'), 'first_f50': str(fp.loc['2007':][fp.loc['2007':] > 0.5].index[0].date()),
            'first_s50': str(sp.loc['2007':][sp.loc['2007':] > 0.5].index[0].date())}


def ms_us_pitfalls():
    """Two specifications that find other regimes: a switching variance on 1947-2019 (the Great Moderation) and a
    switching mean on the sample that includes 2020 (a one-quarter pandemic regime)."""
    g = us_gdp_growth()
    rec = nber('QS')
    x = g.loc[MS_SAMPLE[0]:MS_SAMPLE[1]]
    rv = ms_fit(sm.tsa.MarkovRegression(x.values, k_regimes=2, switching_variance=True))
    hv = int(np.argmax([rv.params[rv.model.param_names.index(f'sigma2[{k}]')] for k in range(2)]))
    sp = pd.Series(rv.smoothed_marginal_probabilities[:, hv], index=x.index)
    calm = sp[sp < 0.5]
    xa = g.loc[MS_SAMPLE[0]:]
    ra = ms_fit(sm.tsa.MarkovRegression(xa.values, k_regimes=2))
    lowa = regime_order(ra)[0]
    spa = pd.Series(ra.smoothed_marginal_probabilities[:, lowa], index=xa.index)
    sv = ms_summary(rv, x.index, 1 - hv)
    return {'sv': sv, 'sv_conc': concordance(pd.Series(rv.smoothed_marginal_probabilities[:, hv], index=x.index), rec),
            'sv_switch': str(calm.index[0].date()) if len(calm) else '', 'sv_hv_share_pre84': float((sp.loc[:'1983'] > 0.5).mean()),
            'sv_hv_share_post84': float((sp.loc['1984':] > 0.5).mean()),
            'all': ms_summary(ra, xa.index, lowa), 'all_low_q': [str(d.date()) for d in spa[spa > 0.5].index]}


def fig_ms_ro(save_it=True):
    """Romanian real GDP growth (q/q): two regimes with switching mean and variance (stable and volatile)."""
    y = ro_gdp()
    g = y.diff().dropna()
    r = ms_fit(sm.tsa.MarkovRegression(g.values, k_regimes=2, switching_variance=True))
    hv = int(np.argmax([r.params[r.model.param_names.index(f'sigma2[{k}]')] for k in range(2)]))
    sp = pd.Series(r.smoothed_marginal_probabilities[:, hv], index=g.index)
    fig, axes = plt.subplots(2, 1, figsize=(10, 5.2), sharex=True)
    axes[0].bar(g.index, g.values, width=70, color=st.MainBlue, label='real GDP growth, % q/q')
    axes[0].axhline(0, color=st.DarkText, lw=0.6)
    axes[1].fill_between(sp.index, 0, sp.values, color=st.IDAred, alpha=0.35, lw=0, step='mid', label='_')
    axes[1].plot(sp.index, sp.values, color=st.IDAred, lw=1.5, drawstyle='steps-mid', label='smoothed Pr(volatile regime)')
    axes[1].set_ylim(-0.02, 1.02)
    st.fig_legend_bottom(fig, ncol=2, y=0.05)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    save('tsa_ch10_ms_ro', save_it)
    s = ms_summary(r, g.index, hv)
    vol = sp > 0.5
    spells, cur = [], None
    for d, v in vol.items():
        if v and cur is None:
            cur = d
        if not v and cur is not None:
            spells.append((str(cur.date()), str(prev.date())))
            cur = None
        prev = d
    if cur is not None:
        spells.append((str(cur.date()), str(prev.date())))
    s.update({'spells': spells, 'share_vol': float(vol.mean()), 'first': str(g.index[0].date()), 'last': str(g.index[-1].date()),
              'sd1': float(np.sqrt(s['sigma2_1'])), 'sd2': float(np.sqrt(s['sigma2_2'])), 'p_last': float(sp.iloc[-1])})
    return s

## Run

In [ ]:
print(fig_ms_us())
print(fig_ms_filtered())
print(ms_us_pitfalls())
print(fig_ms_ro())

![tsa_ch10_ms_us](./tsa_ch10_ms_us.png)

Output: `tsa_ch10_ms_us.pdf`

![tsa_ch10_ms_filtered](./tsa_ch10_ms_filtered.png)

Output: `tsa_ch10_ms_filtered.pdf`

![tsa_ch10_ms_ro](./tsa_ch10_ms_ro.png)

Output: `tsa_ch10_ms_ro.pdf`